# <div align='center'>🌍 Cidades Inteligentes: Qualidade do Ar em Bogotá (RMCAB) </div>

![Banner](https://raw.githubusercontent.com/JamaicaStoAndre/Ci-ncia-de-dados-com-Pandas/master/bogota_air_quality_banner.png)

--- 

## 🎓 Contexto: IA na Borda - Mestrado UFSC

Este material faz parte do projeto de monitoramento da **Rede de Monitoramento da Qualidade do Ar de Bogotá (RMCAB)**. Aqui, exploramos a **Biblioteca Pandas** como a ferramenta central da **Fase B (Design e Preparação)** da nossa arquitetura padronizada de Ciência de Dados.

### 📋 Objetivos de Aprendizagem
1. **Dominar o Pandas**: Manipulação de Series e DataFrames.
2. **Ingestão Inteligente**: Leitura de CSV e extração de metadados.
3. **Análise Didática**: Filtros, GroupBy e EstatísticasDescritivas.
4. **Ponte para IA**: Preparação de dados (One-Hot & Temporal) para a **Fase C**.

In [ ]:
import pandas as pd           # Biblioteca principal para manipulação de tabelas
import numpy as np            # Suporte matemático e para valores nulos (NaN)
import matplotlib.pyplot as plt # Geração de gráficos básicos
import seaborn as sns          # Visualizações estatísticas elegantes
import folium                 # Mapas interativos
from folium.plugins import HeatMap # Plugin de calor para mapas
import re                     # Expressões regulares para limpeza de strings
import os                     # Interação com o sistema de arquivos

# --- FUNÇÕES UTILITÁRIAS DO PROJETO ---

def parse_dms(coor):
    """
    Converte coordenadas no formato DMS (Graus, Minutos, Segundos) para Decimal. 
    Essencial para plotar os sensores no mapa do Folium.
    """
    if not isinstance(coor, str): return coor
    
    # re.split usa raw string r'' para evitar avisos de escape sequence
    parts = re.split(r'[^\d\w]+', coor)
    
    degrees = float(parts[0])               # Graus
    minutes = float(parts[1])               # Minutos
    seconds = float(parts[2]+'.'+parts[3])  # Segundos
    direction = parts[4]                    # N, S, E, W
    
    dec_coord = degrees + minutes / 60 + seconds / 3600
    
    # Se for Sul ou Oeste, a coordenada decimal deve ser negativa
    if direction in ['S', 'W']:
        dec_coord *= -1
    return dec_coord

print("✅ Bibliotecas carregadas e funções utilitárias (DMS to Decimal) prontas!")

## 📂 1. Ingestão de Dados (Fase B - Design)

Nesta etapa, conectamos nossa fonte de dados. O Pandas suporta CSV, Excel, SQL e até JSON. Estamos usando uma amostra real da rede RMCAB.

In [ ]:
# --- CONFIGURAÇÃO DE AMBIENTE ---
# Mude para True se você montou seu Google Drive no Colab
USE_DRIVE = False 

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    csv_path = '/content/drive/MyDrive/1-MestradoUFSC/IA_NA_BORDA/bogota_sensors_sample.csv'
else:
    # Caso contrário, busque no diretório local do Colab (upload manual)
    csv_path = 'bogota_sensors_sample.csv'

# Verificação e Carga
if os.path.exists(csv_path):
    # pd.read_csv transforma o arquivo em um DataFrame (Tabela interativa)
    df = pd.read_csv(csv_path)
    
    # Limpeza: Aplicamos nossa função de coordenadas em todas as linhas
    df['Lat_Dec'] = df['Latitude'].apply(parse_dms)
    df['Lon_Dec'] = df['Longitude'].apply(parse_dms)
    
    print(f"🚀 Sucesso! Carregamos {len(df)} registros de qualidade do ar.")
    display(df.head()) # Exibe as primeiras 5 linhas
else:
    print(f"❗ ERRO: O arquivo '{csv_path}' não foi encontrado.")
    print("DICA: Faça o upload manual de 'bogota_sensors_sample.csv' no Colab.")

## 📊 2. Análise Exploratória (Pandas Fundamentals)

Antes de aplicar IA, precisamos entender os números. O Pandas facilita o cálculo de médias, medianas e desvios por grupo.

In [ ]:
if 'df' in locals():
    # .describe() gera um resumo estatístico instantâneo
    print("--- Estatísticas de Poluentes (PM2.5, PM10, CO) ---")
    display(df[['PM2.5', 'PM10', 'CO']].describe())

    # .groupby() permite agrupar dados por categorias (neste caso, por Estação de Monitoramento)
    print("\n--- Média de Poluição por Estação (GroupBy) ---")
    mean_pollution = df.groupby('Station')[['PM2.5', 'PM10']].mean().reset_index()
    # Ordenando para ver quem polui mais primeiro
    mean_pollution = mean_pollution.sort_values(by='PM2.5', ascending=False)
    display(mean_pollution)

## ⚠️ 3. Filtros Inteligentes (Alertas em Saúde)

O filtro é uma das funções mais poderosas do Pandas. Imagine identificar áreas em estado de emergência (PM2.5 > 50 µg/m³) instantaneamente.

In [ ]:
if 'df' in locals():
    # Criamos uma máscara de seleção
    sensor_em_alerta = df['PM2.5'] > 50
    
    # Aplicamos ao dataframe original
    df_alerta = df[sensor_em_alerta]
    
    print(f"🚨 Atenção! Identificamos {len(df_alerta)} episódios de poluição crítica.")
    display(df_alerta[['DateTime', 'Station', 'PM2.5']])

## 🤖 4. Preparação para IA (Ponte para Fase C)

Modelos de IA não leem texto ou datas puras. O Pandas converte esses dados para formatos numéricos e temporais que as redes neurais entendem.

In [ ]:
if 'df' in locals():
    # 1. Tratamento Temporal: Converter string para Objeto Datetime
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df['hora'] = df['DateTime'].dt.hour
    df['dia_semana'] = df['DateTime'].dt.dayofweek

    # 2. One-Hot Encoding: Transforma categorias ('Estação') em colunas zero/um (binárias)
    # Essencial para redes neurais e modelos de regressão
    df_preparado = pd.get_dummies(df, columns=['Station'], prefix='Estacao')

    print("🛠️ Dataset transformado e pronto para treinamento de Machine Learning:")
    display(df_preparado.head())

## 🗺️ 5. Impacto Final: Visualização Geoespacial

Este é o resultado final da nossa jornada: converter dados brutos de sensores em um mapa de calor interativo para tomada de decisão.

In [ ]:
if 'df' in locals():
    # Criando o mapa base centrado no centro geográfico das estações
    mapa_bogota = folium.Map(location=[4.65, -74.1], zoom_start=12, tiles='cartodbpositron')

    # Preparando os dados de calor [Latitude, Longitude, Intensidade (PM2.5)]
    camada_calor = [[row['Lat_Dec'], row['Lon_Dec'], row['PM2.5']] for index, row in df.iterrows()]

    # Adicionando a camada ao mapa
    HeatMap(camada_calor, radius=20, blur=15, gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'}).add_to(mapa_bogota)
    
    print("🗺️ Mapa de Risco Gerado! Observe as zonas de maior concentração em vermelho.")
    display(mapa_bogota)

---

## 🎓 Desafio Prático (Mãos à Obra!)

Agora é a sua vez! Tente realizar estas duas tarefas rápidas no Pandas:

1.  **Encontre a Estação com o ar mais limpo**: Qual foi o valor **mínimo** registrado de PM10 e em qual estação?
2.  **Filtro Direcionado**: Filtre apenas as mensagens da estação **'TUN'** (Tunal) que tenham PM2.5 maior que 30.

*DICA: Use `df.min()` ou `df.groupby(...).min()` para o desafio 1!*